In [1]:
import numpy as np
import pandas as pd
import io
import scipy as sp
from scipy.stats import qmc
from numpy.random import Generator, PCG64

# Individual TE data

In [2]:
folder = '../../Data/'
df_TEs = pd.read_excel(folder+'TEsummary.xlsx',header=[0],index_col=0)
df_TEs.head()

# T_lim = df_TEs['Threshold (°C) – Est.'].to_numpy()
# beta = df_TEs['Beta (1/°C)'].to_numpy()

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [13]:
print(beta)

[ 4.604503  2.656444  3.837086  4.604503  9.866793  6.906755 13.81351
 13.81351   8.633443]


In [9]:
print(df_TEs.index.to_numpy())

['AMAZ' 'AMOC' 'ASI' 'ENSO' 'GIS' 'PERM' 'REEF' 'WAIS' 'WAM']


# Generating interaction matrices

In [6]:
import scipy as sp
from scipy.stats import qmc
from numpy.random import Generator, PCG64

no_samples = 500
no_edges = 17
seed = 45643564852856852352111
sampler = qmc.LatinHypercube(d = no_edges,rng=Generator(PCG64(seed))) # 17 edges
sample = sampler.random(n=no_samples) # still in a [0,1]^17 hypercube 500 samples!
TEs = ['AMAZ', 'AMOC', 'ASI', 'ENSO', 'GIS', 'PERM', 'REEF', 'WAIS', 'WAM'] # see README.txt
TE_to_ind = {te:i for i,te in enumerate(TEs)}

folder = '../Data/'
df_interactions = pd.read_excel(folder+'Interactions_TableA2.xlsx',usecols='A:F',header=[0])
# df_interactions.head()

a = df_interactions['min'].to_numpy()
b = df_interactions['max'].to_numpy()
A = []
for r in range(no_samples):
    weights = a+sample[r,:]*(b-a)
    mat = np.zeros([len(TEs)]*2)
    for index, row in df_interactions.iterrows():
        i = TE_to_ind[row['Source']]
        j = TE_to_ind[row['Target']]
        mat[i,j] = weights[index]
    A.append(mat)
    # np.savetxt("../Data/Interaction matrices/A_%s.csv"%r,A[r])

# Temperature

In [5]:
df = pd.read_excel('../Data/extendedSSPs.xlsx')
scenarios = df['scenario'].unique()
dfs = {s: df[df['scenario']==s].set_index('quantile').iloc[:,9:].transpose() for s in scenarios}

# Plotting map of TEs

In [4]:
from mpl_toolkits.basemap import Basemap
import networkx as nx

In [ ]:
%run DataProcessingTE.py
# Outputs the following:
    # df_AM22 - processed individual TE data from Armstrong McKay et al 2022
    # df_W24 - TE-TE interactions from Wunderling et al 2024
    # df_SSPs - global mean surface temp. anomalies vs pre-industrial (1850-1900). Historical and several SSPs
        # See scenarios for the list of scenarios 

## NEED TO STREAMLINE TO GET RID OF THIS .py

In [ ]:
G = nx.from_pandas_edgelist(df_W24,source='Source',target='Target',edge_attr=['Dist. Min.', 'Dist. Max.'],create_using=nx.DiGraph)
m = Basemap(projection='robin',lon_0=-30,resolution='c');
pos_lonlat = {'GIS':[-43,70], 'AMOC':[-30,43],'ENSO':[-130,-8.8],
              'WAIS':[-90,-75],'ASI':[5,70],'PERM':[100,60],
              'WAM':[14,19.5],'REEF':[-80,12],'AMAZ':[-62.2,-3.5]}
pos_xy ={k:m(v[0],v[1]) for k,v in pos_lonlat.items()}
T_lim_dic = df_AM22['Threshold (°C) - Est.'].to_dict()
T_lim_dic['ENSO'] = 4.5

fig,ax=plt.subplots(figsize=(8,5))

elist = [(e[0],e[1],0) for e in G.edges]
earc_dic = {e:0 for e in G.edges}
earc_dic[('GIS','AMOC')]=0.2
earc_dic[('AMOC','GIS')]=0.2
earc_dic[('AMOC','WAIS')]=-0.5
earc_dic[('AMOC','ENSO')]=0.3
earc_dic[('WAIS','AMOC')]=0.3
earc_dic[('ASI','AMOC')]=0.15
earc_dic[('AMOC','ASI')]=0.15
earc_dic[('ASI','GIS')]=0.1
earc_dic[('PERM','AMOC')]=-0.15
earc_dic[('ASI','PERM')]=-0.1
earc_dic[('WAIS','GIS')]=-1.3
earc_dic[('GIS','WAIS')]=1.6
# e_man_list = [('GIS','WAIS'),('WAIS','GIS'),('ASI','GIS'),('AMOC','WAIS')]
# for e in e_man_list:
#     elist.remove(e)
# m.drawcoastlines();
m.fillcontinents(color='lightgray',lake_color='white');
# x,y = m(pos['GIS'][0],pos['GIS'][1])
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

nx.draw_networkx(G,ax=ax,pos=pos_xy,node_size=400,with_labels=True,edgelist=[],
                 node_color=[T_lim_dic[n] for n in G.nodes()],
                 vmin=-1.5,vmax=7,
                 cmap='YlGn_r',font_size=12,font_color='navy',font_weight=None)
for e in G.edges:
    vals = G.get_edge_data(e[0],e[1])
    if vals['Dist. Min.'] > 0: # Destabilising
        c = 'r'
        w = vals['Dist. Max.']
    elif vals['Dist. Max.'] < 0: # Stabilising
        c = 'b'
        w= np.abs(vals['Dist. Min.'])
    else: # Unclear
        c = 'gray' 
        w = vals['Dist. Max.']
    nx.draw_networkx_edges(G,pos_xy,ax=ax,edgelist=[e], 
                           edge_color=c, width=np.log(w)*1.1,
                           connectionstyle="arc3,rad=rrr".replace('rrr',str(earc_dic[e])));

custom_lines = [Line2D([0], [0], color='r', lw=1, marker='>'),
                Line2D([0], [0], color='b', lw=1, marker='>'),
                Line2D([0], [0], color='gray', lw=1, marker='>')]
ax.legend(custom_lines,['Destabilizing','Stabilizing', 'Unclear'],loc=[0.6,0.1])
# fig.colorbar(im,ax=ax)
cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap='YlGn_r',norm=mpl.colors.Normalize(vmin=-1.5,vmax=7)), 
             ax=ax, orientation='horizontal',fraction=0.05,pad=0.01)
cbar.ax.set_xlim([1,7])
cbar.ax.set_xticklabels([str(r)+'℃' for r in range(1,8)]);